In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import datetime

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import KFold, train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from hyperopt import STATUS_OK, Trials, fmin, hp, tpe

import warnings
warnings.filterwarnings('ignore')

# Extracting the Data

In [ ]:
import os

data_path = 'D:\DataProject\New York Taxi'

os.listdir(data_path)

In [ ]:
trip_df = pd.read_csv(data_path + 'train.zip')
trip_df.head()

In [ ]:
trip_df.shape

In [ ]:
trip_df.info()

In [ ]:
trip_df.describe()

# Exploratory Data Analysis along with Feature Engineering

trip_duration

In [ ]:
print('Average Trip Duration: {} seconds (roughly {} minutes)'.format(
        trip_df['trip_duration'].mean(), round(trip_df['trip_duration'].mean() / 60, 2)
      ))
print('Maximum Trip Duration: {} seconds (roughly {} days)'.format(
        trip_df['trip_duration'].max(), round(((trip_df['trip_duration'].max() / 60) / 60) / 24, 2)
      ))
print('Minimum Trip Duration: {} seconds'.format(trip_df['trip_duration'].min()))

In [ ]:
# Calculating Pearson's Index of Skewness
mean, median, std = trip_df.agg({'trip_duration': ['mean', 'median', 'std']}).values.flatten()

skewness = round(3 * (mean - median) / std, 4)
print("Pearson's Index:", skewness)

In [ ]:
plt.hist(trip_df.trip_duration, bins = 100)
plt.xlim(0.0e5, 0.4e5);

In [ ]:
# Finding the range where `trip_duration` data are more clustered around
no_of_td = trip_df.trip_duration.count()
no_of_td_clustered = trip_df[(trip_df.trip_duration > 100) & (trip_df.trip_duration < 3000)].shape[0]

round((no_of_td_clustered/ no_of_td ) * 100, 2)

In [ ]:
# Visualizing the range where `trip_duration` data are more clustered around
duration_df = trip_df[(trip_df.trip_duration > 100) & (trip_df.trip_duration < 3000)]

ax = sns.histplot(data = duration_df, x = duration_df['trip_duration'] / 60, bins = 30)
ax.set_xlabel('Trip Duration (in Minutes)');

In [ ]:
# Calculating IQR, lower range and upper range
Q3 = np.quantile(trip_df['trip_duration'], 0.75)
Q1 = np.quantile(trip_df['trip_duration'], 0.25)
IQR = Q3 - Q1

lower_range = Q1 - 1.5 * IQR
upper_range = Q3 + 1.5 * IQR

IQR, lower_range, upper_range

In [ ]:
# Calculating the percentage of data left if we remove outliers
trip_df[(trip_df.trip_duration > lower_range) & (trip_df.trip_duration < upper_range)].shape[0] / trip_df.shape[0]

In [ ]:
# Removing outliers that are less than `lower_range` or greater than `upper_range`
trip_df = trip_df[(trip_df.trip_duration > lower_range) & (trip_df.trip_duration < upper_range)].copy(deep = True)

In [ ]:
ax = sns.histplot(data = trip_df, x = 'trip_duration', bins = 50)
ax.set_xlabel('Trip Duration (in Seconds)');

pickup_datetime & dropoff_datetime

In [ ]:
print(trip_df[['pickup_datetime', 'dropoff_datetime']].dtypes)

trip_df[['pickup_datetime', 'dropoff_datetime']].head()

In [ ]:
# Changing datatype: 'object' to 'datetime64'
trip_df['pickup_datetime'] = pd.to_datetime(trip_df['pickup_datetime'])
trip_df['dropoff_datetime'] = pd.to_datetime(trip_df['dropoff_datetime'])

print(trip_df[['pickup_datetime', 'dropoff_datetime']].dtypes)

In [ ]:
# Creating new features using `dropoff_datetime` column
trip_df['year'] = trip_df['dropoff_datetime'].dt.year.astype(int)
trip_df['month'] = trip_df['dropoff_datetime'].dt.month.astype(int)
trip_df['dayofmonth'] = trip_df['dropoff_datetime'].dt.day.astype(int)
trip_df['dayofweek'] = trip_df['dropoff_datetime'].dt.dayofweek.astype(int)

trip_df.head()

year

In [ ]:
trip_df.year.value_counts()

month

In [ ]:
# This helper function is used for `dropoff_month`, `dropoff_day` and `dropoff_dayofweek` variables
# It evaluates "average trip duration" and "number of trips" for each day of month or week
# and return the dataframe
def aggregate_df(feature):
    features = {'month': 'Month',
                'dayofmonth': 'Day of Month',
                'dayofweek': 'Day of Week'}

    df = pd.DataFrame(trip_df.groupby(feature).mean('trip_duration')['trip_duration'])
    df.index.name = features.get(feature)
    df.rename(columns = {'trip_duration': 'Average Trip Duration'}, inplace = True)
    df['Number of Trips'] = trip_df[feature].value_counts()
    df.reset_index(inplace = True)

    return df

In [ ]:
month_df = aggregate_df('month')
month_df

In [ ]:
ax = sns.barplot(data = month_df, x = 'Month', y = 'Number of Trips',
                 color = 'blue', alpha = 0.5)

ax2 = ax.twinx()
ax2 = sns.lineplot(data = month_df, x = 'Month', y = 'Average Trip Duration',
                   color = 'red', label = 'Average Trip Duration');

day of month

In [ ]:
day_df = aggregate_df('dayofmonth')
day_df.head()

In [ ]:
# Visualizing "Number of Trips" and "Average Tip Duration" of Day of Month
plt.figure(figsize = (7, 4))

ax = sns.lineplot(data = day_df, x = 'Day of Month', y = 'Number of Trips', label = 'Number of Trips')
ax.set(ylabel = None)

ax2 = ax.twinx()
sns.lineplot(data = day_df, x = 'Day of Month', y = 'Average Trip Duration', ax = ax2, color = 'r', label = 'Average Trip Duration')
ax2.set(ylabel = None)

ax.legend(loc = 'upper left', bbox_to_anchor = (0.05, 1.15))
ax2.legend(loc = 'upper right', bbox_to_anchor = (0.95, 1.15));

day of week

In [ ]:
day_week_df = aggregate_df('dayofweek')
day_week_df

In [ ]:
ax = sns.barplot(data = day_week_df, x = 'Day of Week', y = 'Number of Trips',
                 color = 'blue', alpha = 0.5)

ax2 = ax.twinx()
sns.barplot(data = day_week_df, x = 'Day of Week', y = 'Average Trip Duration',
            ax = ax2, linewidth = 2, edgecolor = '.5', facecolor = (0, 0, 0, 0))

sns.despine(left = True);

Part of Day

In [ ]:
def define_part_of_day():
    hours = trip_df['pickup_datetime'].dt.hour

    conditions = [
        (hours >= 5) & (hours < 12),
        (hours >= 12) & (hours < 17),
        (hours >= 17) & (hours < 21),
        (hours >= 21) | (hours < 5)
    ]

    values = ['morning', 'afternoon', 'evening', 'night']

    trip_df['part_of_day'] = np.select(conditions, values)

define_part_of_day()

In [ ]:
trip_df.head()

In [ ]:
sns.countplot(data = trip_df, x = 'part_of_day', color = 'blue', alpha = 0.5);

passenger_count

In [ ]:
plt.figure(figsize = (7, 4))

no_of_passengers = trip_df.passenger_count.value_counts().sort_index().index
no_of_trips = trip_df.passenger_count.value_counts()
avg_trip_duration = trip_df.groupby('passenger_count').mean()['trip_duration']

# Average Trip Duration
ax = sns.barplot(x = no_of_passengers, y = avg_trip_duration, label = 'Average Trip Duration', color = 'red', alpha = 0.5)
ax.set_xticks(no_of_passengers)
ax.set(xlabel = 'Passenger Count', ylabel = 'Average Trip Duration')

# Number of Trips
ax2 = ax.twinx()
ax2 = sns.lineplot(x = no_of_passengers, y = no_of_trips, label = 'Number of Trips')
ax2.set(ylabel = 'Number of Trips')

ax.legend(loc = 'upper left', bbox_to_anchor = (0.05, 1.15))
ax2.legend(loc = 'upper right', bbox_to_anchor = (0.95, 1.15));

store_and_fwd_flag

In [ ]:
trip_df.store_and_fwd_flag.value_counts() / trip_df.shape[0]

In [ ]:
# Haversine Formula
def distance(lat1, lon1, lat2, lon2):
    p = np.pi / 180
    a = 0.5 - np.cos((lat2 - lat1) * p) / 2 + \
        np.cos(lat1 * p) * np.cos(lat2 * p) * (1 - np.cos((lon2 - lon1) * p)) / 2

    return 12742 * np.arcsin(np.sqrt(a))

In [ ]:
lat1 = np.array(trip_df['pickup_latitude'])
lon1 = np.array(trip_df['pickup_longitude'],)
lat2 = np.array(trip_df['dropoff_latitude'])
lon2 = np.array(trip_df['dropoff_longitude'])

trip_df['distance'] = distance(lat1, lon1, lat2, lon2)
trip_df.head()

In [ ]:
# Finding the range where `distance` is more clustered around
trip_df[(trip_df.distance >= 0) & (trip_df.distance <= 15)].count()[0] / trip_df.shape[0]

In [ ]:
# Visualizing the `distance` using histogram
df = trip_df[(trip_df.distance >= 0) & (trip_df.distance <= 15)]

sns.histplot(data = df, x = 'distance', bins = 30);

# Data Preparation

## Defining Input and Output Dataframes

In [ ]:
print(trip_df.columns)

In [ ]:
X = trip_df.drop(columns = ['id', 'pickup_datetime', 'dropoff_datetime',
                            'pickup_longitude', 'pickup_latitude', 'dropoff_longitude',
                            'dropoff_latitude', 'year', 'trip_duration'])
X.head()

In [ ]:
y = trip_df[['trip_duration']]
y.head()

In [ ]:
# Separating numerical and categorical columns
numerical_columns = X.select_dtypes(include = np.number).columns.tolist()
categorical_columns = X.select_dtypes(include = 'object').columns.tolist()

print('Numerical columns are:', numerical_columns)
print('Categorical columns are:', categorical_columns)

## Preprocessing and Training the Model using Scikit-learn Pipeline

In [ ]:
# Preprocessing for numerical variables
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy = 'median')),
    ('scaler', StandardScaler())
])

# Preprocessing for categorical variables
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy = 'constant', fill_value = 'missing')),
    ('encoder', OneHotEncoder(handle_unknown = 'ignore', sparse = False))
])

# Combining `num_pipe` and `cat_pipe` with ColumnTransfomer
preprocessor = ColumnTransformer([
    ('numerical', num_pipe, numerical_columns),
    ('categorical', cat_pipe, categorical_columns)
])

# Model pipeline
pipe = Pipeline(steps = [
    ('preprocessor', preprocessor),
    ('model', XGBRegressor(booster = 'gbtree', njobs = -1, verbosity = 0))
])

Using KFold Cross Validation to Train and Test the Model

In [ ]:
kfold = KFold(n_splits = 5, shuffle = True, random_state = 42)
fold_num = 1
scores = list()

for train_index, test_index in kfold.split(X, y):
    X_train, y_train = X.iloc[train_index], y.iloc[train_index]
    X_test, y_test = X.iloc[test_index], y.iloc[test_index]

    pipe.fit(X_train, y_train)
    test_pred = pipe.predict(X_test)
    score = mean_squared_error(y_test, test_pred, squared = False)
    scores.append(score)

    print(f"Loss for fold {fold_num}: ", score)
    fold_num += 1

print('--------------------------------')
print('Average Loss:', round(np.mean(scores), 2))

## Hyperparameter Tuning

In [ ]:
def tune_xgb(**params):
    pipe = Pipeline(steps = [
        ('preprocessor', preprocessor),
        ('model', XGBRegressor(booster = 'gbtree', njobs = -1, verbosity = 0, **params))
    ])

    pipe.fit(X_train, y_train)
    test_pred = pipe.predict(X_test)
    rmse = mean_squared_error(y_test, test_pred, squared = False)

    return rmse

In [ ]:
tune_xgb(max_depth = 10)

In [ ]:
tune_xgb(max_depth = 10, min_child_weight = 2)

In [ ]:
# p_loss = tune_xgb(n_estimators = 100)

# for n in range(150, 500, 50):
#     n_loss = tune_xgb(n_estimators = n)
#     print(f"Number of Trees: {n} -> Loss: {n_loss}")

#     if n_loss > p_loss:
#         print('Stopped the program due to overfitting.')
#         break
#     else:
#         p_loss = n_loss

In [ ]:
tune_xgb(n_estimators = 100,
         max_depth = 10,
         min_child_weight = 2,
         eta = 0.3)

In [ ]:
# tune_xgb(n_estimators = 150,
#          max_depth = 10,
#          min_child_weight = 2,
#          eta = 0.3,
#          colsample_bytree = 0.5,
#          colsample_bylevel = 0.5)

In [ ]:
tune_xgb(n_estimators = 100,
         max_depth = 10,
         min_child_weight = 2,
         eta = 0.3,
         colsample_bytree = 0.5,
         colsample_bylevel = 0.5)

Hyperparameter Tuning with RandomizedSearchCV

In [ ]:
# (X_train, X_test), (y_train, y_test) = train_test_split(X, y, test_size = 0.25, random_state = 42)

# preprocessor.fit(X_train)
# cat_columns = preprocessor.named_transformers_['categorical']['encoder'].get_feature_names_out(categorical_columns)
# columns = np.append(numerical_columns, cat_columns)

# X_train = pd.DataFrame(preprocessor.fit_transform(X_train), columns = columns)
# X_test = pd.DataFrame(preprocessor.transform(X_test), columns = columns)

In [ ]:
# params = {'max_depth': [3, 6, 10, 15],
#           'learning_rate': [0.01, 0.1, 0.2, 0.3, 0.4],
#           'subsample': np.arange(0.5, 1.0, 0.1),
#           'colsample_bytree': np.arange(0.5, 1.0, 0.1),
#           'colsample_bylevel': np.arange(0.5, 1.0, 0.1),
#           'n_estimators': [100, 250, 500, 750],
#          }

# xgb = XGBRegressor(booster = 'gbtree', njobs = -1, verbosity = 0)
# model = RandomizedSearchCV(estimator = xgb, param_distributions = params,
#                            scoring = 'neg_root_mean_squared_error',
#                            n_iter = 25, n_jobs = -1)
# model.fit(X_train, y_train)

Hyperparameter Tuning with HYPEROPT

In [ ]:
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 42)

# preprocessor.fit(X_train)
# cat_columns = preprocessor.named_transformers_['categorical']['encoder'].get_feature_names_out(categorical_columns)
# columns = np.append(numerical_columns, cat_columns)

# X_train = pd.DataFrame(preprocessor.fit_transform(X_train), columns = columns)
# X_test = pd.DataFrame(preprocessor.transform(X_test), columns = columns)

# space = {'max_depth': hp.quniform("max_depth", 3, 15, 1),
#          'gamma': hp.uniform ('gamma', 1, 9),
#          'reg_alpha' : hp.quniform('reg_alpha', 40, 180,1),
#          'reg_lambda' : hp.uniform('reg_lambda', 0,1),
#          'colsample_bytree' : hp.uniform('colsample_bytree', 0.5, 1),
#          'min_child_weight' : hp.quniform('min_child_weight', 0, 10, 1),
#          'n_estimators': 100
#          }

# def objective(space):
#     reg = XGBRegressor(n_estimators = space['n_estimators'],
#                        max_depth = int(space['max_depth']),
#                        gamma = space['gamma'],
#                        reg_alpha = int(space['reg_alpha']),
#                        min_child_weight = int(space['min_child_weight']),
#                        colsample_bytree = int(space['colsample_bytree'])
#                        )

#     evaluation = [(X_train, y_train), (X_test, y_test)]

#     reg.fit(X_train, y_train,
#             eval_set = evaluation, eval_metric = "rmse",
#             early_stopping_rounds = 10, verbose = False)


#     test_pred = reg.predict(X_test)
#     loss = mean_squared_error(y_test, test_pred, squared = False)
#     print ("RMSE:", loss)

#     return {'loss': loss, 'status': STATUS_OK }

# trials = Trials()

# best_hyperparams = fmin(fn = objective,
#                         space = space,
#                         algo = tpe.suggest,
#                         max_evals = 100,
#                         trials = trials)

# print("The best hyperparameters are : ","\n")
# print(best_hyperparams)

# Testing the Model and Saving the Results

In [ ]:
test_df = pd.read_csv(data_path + 'test.zip')
test_df.head()

In [ ]:
    def evaluate_model(df):

    """Feature Engineering"""
    df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])

    # Creating new columns using datetime
    df['year'] = df['pickup_datetime'].dt.year.astype(int)
    df['month'] = df['pickup_datetime'].dt.month.astype(int)
    df['dayofmonth'] = df['pickup_datetime'].dt.day.astype(int)
    df['dayofweek'] = df['pickup_datetime'].dt.dayofweek.astype(int)

    # Creating `part_of_day` feature
    hours = df['pickup_datetime'].dt.hour

    conditions = [
        (hours >= 5) & (hours < 12),
        (hours >= 12) & (hours < 17),
        (hours >= 17) & (hours < 21),
        (hours >= 21) | (hours < 5)
    ]
    values = ['morning', 'afternoon', 'evening', 'night']

    df['part_of_day'] = np.select(conditions, values)

    # Creating `distance` column using Haversine Formula
    def distance(lat1, lon1, lat2, lon2):
        p = np.pi / 180
        a = 0.5 - np.cos((lat2 - lat1) * p) / 2 + \
            np.cos(lat1 * p) * np.cos(lat2 * p) * (1 - np.cos((lon2 - lon1) * p)) / 2

        return 12742 * np.arcsin(np.sqrt(a))


    lat1 = np.array(df['pickup_latitude'])
    lon1 = np.array(df['pickup_longitude'],)
    lat2 = np.array(df['dropoff_latitude'])
    lon2 = np.array(df['dropoff_longitude'])

    df['distance'] = distance(lat1, lon1, lat2, lon2)

    ########################################################################

    """Data Preparation"""
    X_test = df.drop(columns = ['id', 'pickup_datetime', 'pickup_longitude', 'pickup_latitude',
                                'dropoff_longitude', 'dropoff_latitude', 'year'])

    numerical_columns = X_test.select_dtypes(include = np.number).columns.tolist()
    categorical_columns = X_test.select_dtypes(include = 'object').columns.tolist()

    ########################################################################

    """Data Preprocessing and Training the Model using Scikit-learn Pipeline"""
    # Preprocessing for numerical variable
    num_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy = 'median')),
        ('scaler', StandardScaler())
    ])

    # Preprocessing for categorical variables
    cat_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy = 'constant', fill_value = 'missing')),
        ('encoder', OneHotEncoder(handle_unknown = 'ignore', sparse = False))
    ])

    # Combining `num_pipe` and `cat_pipe` with ColumnTransfomer
    preprocessor = ColumnTransformer([
        ('numerical', num_pipe, numerical_columns),
        ('categorical', cat_pipe, categorical_columns)
    ])

    # Model pipeline
    # 修改后的 Pipeline 代码
    pipe = Pipeline(steps = [
        ('preprocessor', preprocessor),
        ('model', XGBRegressor(booster = 'gbtree',
                               n_estimators = 100,
                               max_depth = 10,
                               min_child_weight = 2,
                               eta = 0.3,
                               colsample_bytree = 0.5,
                               colsample_bylevel = 0.5,
                               n_jobs = -1,
                               objective = 'reg:squarederror',
                               verbosity = 0))
    ])
    pipe.fit(X, y)
    pred = pipe.predict(X_test)

df['trip_duration'] = pred

return df

In [ ]:
df = evaluate_model(test_df)
df.head()

In [ ]:
submission_df = pd.read_csv(data_path + 'sample_submission.zip')
submission_df.head()

In [ ]:
submission_df['trip_duration'] = df['trip_duration']
submission_df.head()

In [ ]:
# submission_df.to_csv(data_path + 'submission.csv', index = False)